In [0]:
%sql
CREATE OR REPLACE TEMP VIEW bronze_parsed AS
SELECT well_id, sensor_timestamp, wellhead_pressure_raw, flow_bpd_raw, source_file, ingested_at,
       TRY_TO_TIMESTAMP(sensor_timestamp, 'yyyy-MM-dd HH:mm:ss') AS reading_ts,
       TRY_CAST(wellhead_pressure_raw AS DOUBLE) AS pressure_psi,
       TRY_CAST(flow_bpd_raw AS DOUBLE) AS flow_bpd
FROM workspace.oil_gas_bronze.telemetry_raw;

In [0]:
%sql
SELECT COUNT(*) AS total,
       SUM(CASE WHEN pressure_psi IS NULL THEN 1 ELSE 0 END) AS null_pressure,
       SUM(CASE WHEN pressure_psi < 0 THEN 1 ELSE 0 END) AS negative_pressure
FROM bronze_parsed;

In [0]:
%sql
SELECT TRY_CAST('1202.79' AS DOUBLE) AS good,
       TRY_CAST('ERR_VAL_99' AS DOUBLE) AS bad,
       TRY_CAST('' AS DOUBLE) AS empty_text;

In [0]:
%sql
-- We use TRY_CAST instead of CAST because CAST fails the whole job when it finds malformed text like “ERR_VAL_99”. TRY_CAST returns NULL for that value instead, so the job keeps running and I can route the bad rows later.
-- SELECT CAST('ERR_VAL_99' AS DOUBLE); [CAST_INVALID_INPUT] The value 'ERR_VAL_99' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
-- == SQL (line 1, position 8) ==
-- SELECT CAST('ERR_VAL_99' AS DOUBLE)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW bronze_classified AS
SELECT *,
  CASE
    WHEN well_id IS NULL OR well_id = '' THEN 'missing_well_id'
    WHEN reading_ts IS NULL THEN 'invalid_timestamp'
    WHEN pressure_psi IS NULL THEN 'invalid_pressure'
    WHEN pressure_psi < 0 THEN 'negative_pressure'
    WHEN flow_bpd IS NULL OR flow_bpd < 0 THEN 'invalid_flow'
  END AS reject_reason
FROM bronze_parsed;

In [0]:
%sql
SELECT reject_reason, COUNT(*) AS rows_n
FROM bronze_classified
GROUP BY reject_reason
ORDER BY rows_n DESC;

In [0]:
%sql
MERGE INTO workspace.oil_gas_silver.telemetry_quarantine AS t
USING (
  SELECT well_id, sensor_timestamp, wellhead_pressure_raw, flow_bpd_raw, reject_reason, source_file
  FROM bronze_classified
  WHERE reject_reason IS NOT NULL
) AS s
ON t.source_file = s.source_file
   AND t.well_id <=> s.well_id
   AND t.sensor_timestamp <=> s.sensor_timestamp
WHEN NOT MATCHED THEN INSERT
  (well_id, sensor_timestamp, wellhead_pressure_raw, flow_bpd_raw, reject_reason, source_file, quarantined_at)
VALUES
  (s.well_id, s.sensor_timestamp, s.wellhead_pressure_raw, s.flow_bpd_raw, s.reject_reason, s.source_file, current_timestamp());

In [0]:
%sql
SELECT reject_reason, COUNT(*) AS rows_n
FROM workspace.oil_gas_silver.telemetry_quarantine
GROUP BY reject_reason;

In [0]:
%sql
MERGE INTO workspace.oil_gas_silver.telemetry_clean AS t
USING (
  SELECT well_id, reading_ts, pressure_psi, flow_bpd, source_file
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY well_id, reading_ts ORDER BY ingested_at DESC) AS rn
    FROM bronze_classified
    WHERE reject_reason IS NULL
  )
  WHERE rn = 1
) AS s
ON t.well_id = s.well_id AND t.reading_ts = s.reading_ts
WHEN MATCHED AND (t.pressure_psi <> s.pressure_psi OR t.flow_bpd <> s.flow_bpd) THEN
  UPDATE SET t.pressure_psi = s.pressure_psi, t.flow_bpd = s.flow_bpd,
             t.source_file = s.source_file, t.silver_loaded_at = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (well_id, reading_ts, pressure_psi, flow_bpd, source_file, silver_loaded_at)
  VALUES (s.well_id, s.reading_ts, s.pressure_psi, s.flow_bpd, s.source_file, current_timestamp());

In [0]:
%sql
SELECT (SELECT COUNT(*) FROM workspace.oil_gas_silver.telemetry_clean) AS clean_rows,
       (SELECT COUNT(*) FROM workspace.oil_gas_silver.telemetry_quarantine) AS quarantined,
       (SELECT COUNT(*) FROM workspace.oil_gas_bronze.telemetry_raw) AS bronze_rows;